### Unified errors
This notebook produces a dataframe with the attributes (Benchmark, prompt/problem_id, model, library, generated code, message errors, and error type) from the different benchmarks.

In [3]:
import json
import re
import pandas as pd

In [4]:
def clean_error_msg(error_msg):
    """
        Retrieves error line and exception
    """
    if error_msg:
        text = 'line'
        pattern = re.compile(r'\b' + re.escape(text) + r'\b')
        lines = [match.start() for match in pattern.finditer(error_msg)]
        if len(lines) > 0:
            return error_msg[lines[-1]:]
    return ""

In [82]:
def read_jsonl(file_name, cols, unlist_cols, left_cols):
    """
        Returns a dataframe with chosen column names
        and columns to unlist
    """
    df_normalized = None
    normalized_dfs = []
    with open(file_name, 'r') as file:
        for line in file:
            data_line = json.loads(line)
            if len(unlist_cols) > 0:
                for column in unlist_cols:
                    normalized_column = pd.json_normalize(data_line[column])
                    for l in left_cols:
                        normalized_column[l] = data_line[l]
                    normalized_dfs.append(normalized_column)
                    # Concatenate the normalized DataFrames
                    df_normalized = pd.concat(normalized_dfs, axis=0, ignore_index=True)
            else:
                normalized_dfs.append(data_line)
                # Transform list to df
                df_normalized = pd.DataFrame(normalized_dfs)
    return df_normalized[cols]

In [6]:
def classify_error_msg(df):
    """
        Adds types of errors in the df
        given an error_message
    """
    # Tab error and Indentation error
    df['contains_indentation'] = df['message'].str.contains('indentation')
    # Syntax error
    df['contains_syntax'] = df['message'].str.contains('syntax')
    # Type Error
    df['contains_unexpected'] = df['message'].str.contains('unexpected')
    # Type or Value Error
    df['contains_must'] = df['message'].str.contains('must')
    # Attribute Error
    df['contains_has'] = df['message'].str.contains('has no')
    # Name Error
    # All elements in defined are in name (defined is a subelement of name)
    df['contains_name'] = df['message'].str.contains('name')
    df['contains_defined'] = df['message'].str.contains('defined')
    return df

In [112]:
def errors_type(error_msg):
    """
        Gets error message
    """
    match = re.search(r'\b\w*Error\w*\b', error_msg)
    
    # If a match is found, return the matched word
    if match:
        return match.group(0)
    elif "Result not True" in error_msg:
        return "AssertionError"
    else:
        match = re.search(r'\b\w*re.error: .*$', error_msg)
        if match:
            return match.group(0)
        match = re.search(r'\b\w*Exception: .*$', error_msg)
        if match:
            return match.group(0)
        if error_msg == '':
            return 'passed'
        return "Other error"

In [ ]:
def get_experiments_by_path(paths=[], benchmark, models=[]):
    lst = []
    for path in paths:
        # specific processing for benchmark
        if benchmark == 'DS-1000':
            df = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
            df["error_type"] = df["result"].apply(errors_type)
        elif benchmark == 'CodeGen':
            df = read_jsonl(path, cols=['project','_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['project','_id'])
            df["message"] = df["eror_msg"].apply(clean_error_msg)
            df["error_type"] = df["message"].apply(errors_type)
            df["error_type"] = df["error_type"].replace('', 'passed')
        lst.append(df)
    return lst

In [ ]:
lst = get_experiments_by_path(paths, benchmark='CodeGen', models)

for 

In [113]:
path = 'Data/CoderEval/CodeGen/Raw_Input.jsonl_out.jsonl'
data_codegen = read_jsonl(path, cols=['project','_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['project','_id'])
data_codegen["message"] = data_codegen["eror_msg"].apply(clean_error_msg)
data_codegen["error_type"] = data_codegen["message"].apply(errors_type)
#classify_error_msg(data_codegen)
data_codegen["error_type"] = data_codegen["error_type"].replace('', 'passed')
data_codegen = (data_codegen.groupby('error_type')['message'].count()).reset_index().rename(columns={"message": "350M CodeGen-Mono"})

In [114]:
path = 'Data/CoderEval/ChatGPT/Raw_Input.jsonl_out.jsonl'
data_chatgpt = read_jsonl(path, cols=['project','_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['project','_id'])
data_chatgpt["message"] = data_chatgpt["eror_msg"].apply(clean_error_msg)
data_chatgpt["error_type"] = data_chatgpt["message"].apply(errors_type)
#classify_error_msg(data_codegen)
data_chatgpt["error_type"] = data_chatgpt["error_type"].replace('', 'passed')
data_chatgpt = (data_chatgpt.groupby('error_type')['message'].count()).reset_index().rename(columns={"message": "gpt-3.5-turbo"})

In [115]:
path = 'Data/CoderEval/PanGu-Coder/Raw_Input.jsonl_out.jsonl'
data_pangu= read_jsonl(path, cols=['_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['_id'])
data_pangu["message"] = data_pangu["eror_msg"].apply(clean_error_msg)
#classify_error_msg(data_pangu)
data_pangu["error_type"] = data_pangu["message"].apply(errors_type)
data_pangu["error_type"] = data_pangu["error_type"].replace('', 'passed')
data_pangu = (data_pangu.groupby('error_type')['message'].count()).reset_index().rename(columns={"message": "300M PanGu-Coder"})

In [116]:
merged_df = pd.merge(data_codegen, data_chatgpt, on='error_type', how='outer')
merged_df = pd.merge(merged_df, data_pangu, on='error_type', how='outer')
merged_df.sort_values(by='gpt-3.5-turbo', ascending=False).to_csv("results_errors_codereval.csv", index=False)

In [119]:
merged_df.iloc[:, 1:] = merged_df.iloc[:, 1:] / 2300

In [120]:
ds_results = pd.read_csv("results_errors.csv")
ds_results.iloc[:, 1:] = ds_results.iloc[:, 1:] / 1000

In [121]:
merged_results = pd.merge(merged_df, ds_results, on='error_type', how='outer')

In [122]:
merged_results.columns

Index(['error_type', '350M CodeGen-Mono', 'gpt-3.5-turbo', '300M PanGu-Coder',
       'gpt-3.5-turbo-0125', 'gpt-3.5-turbo-0613', 'gpt-4-0613',
       'gpt-4-turbo-2024-04-09', 'claude-3-opus-20240229'],
      dtype='object')

In [132]:
import numpy as np
rdf = merged_results[['error_type', 'gpt-3.5-turbo','gpt-3.5-turbo-0125', 'gpt-3.5-turbo-0613']].fillna(0).dropna(how='all').sort_values(by='gpt-3.5-turbo-0125', ascending=False)

In [134]:
rdf['sum'] = rdf['gpt-3.5-turbo']+rdf['gpt-3.5-turbo-0125']+rdf['gpt-3.5-turbo-0613']

In [136]:
rdf.sort_values(by='sum', ascending=False).to_csv("results_errors_gpt35.csv", index=False)

In [87]:
path = 'Data/DS-1000/data/gpt-3.5-turbo-0125-0-errors.jsonl'
data_ds1000_one = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
data_ds1000_one["error_type"] = data_ds1000_one["result"].apply(errors_type)
#data_ds1000_one = (data_ds1000_one.groupby('error_type')['result'].count()).reset_index().rename(columns={"result": "gpt-3.5-turbo-0125"})

In [63]:
data_ds1000_one

,result,library,perturbation_type,error_type
0,"failed: 'result', type: <class 'KeyError'>",Pandas,Origin,KeyError
1,"failed: , type: <class 'AssertionError'>",Pandas,Origin,AssertionError
2,"failed: 'result', type: <class 'KeyError'>",Pandas,Difficult-Rewrite,KeyError
3,passed,Pandas,Surface,passed
4,"failed: 'result', type: <class 'KeyError'>",Pandas,Semantic,KeyError
...,...,...,...,...
995,failed: Expected size for first two dimensions...,Pytorch,Origin,RuntimeError
996,failed: Sizes of tensors must match except in ...,Pytorch,Semantic,RuntimeError
997,"failed: , type: <class 'AssertionError'>",Pytorch,Origin,AssertionError
998,failed: Dimension out of range (expected to be...,Pytorch,Origin,IndexError


In [51]:
path = 'Data/DS-1000/data/gpt-3.5-turbo-0613-0-errors.jsonl'
data_ds1000_two = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
data_ds1000_two["error_type"] = data_ds1000_two["result"].apply(errors_type)
data_ds1000_two = (data_ds1000_two.groupby('error_type')['result'].count()).reset_index().rename(columns={"result": "gpt-3.5-turbo-0613"})

In [52]:
path = 'Data/DS-1000/data/gpt-4-0613-0-errors.jsonl'
data_ds1000_three = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
data_ds1000_three["error_type"] = data_ds1000_three["result"].apply(errors_type)
data_ds1000_three = (data_ds1000_three.groupby('error_type')['result'].count()).reset_index().rename(columns={"result": "gpt-4-0613"})

In [53]:
path = 'Data/DS-1000/data/gpt-4-turbo-2024-04-09-0-errors.jsonl'
data_ds1000_four = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
data_ds1000_four["error_type"] = data_ds1000_four["result"].apply(errors_type)
data_ds1000_four = (data_ds1000_four.groupby('error_type')['result'].count()).reset_index().rename(columns={"result": "gpt-4-turbo-2024-04-09"})

In [54]:
path = 'Data/DS-1000/data/claude-3-opus-20240229-0-errors.jsonl'
data_ds1000_claude = read_jsonl(path, cols=['result','library','perturbation_type'], unlist_cols=[], left_cols=['result','library','perturbation_type'])
data_ds1000_claude["error_type"] = data_ds1000_claude["result"].apply(errors_type)
data_ds1000_claude = (data_ds1000_claude.groupby('error_type')['result'].count()).reset_index().rename(columns={"result": "claude-3-opus-20240229"})

In [55]:
merged_df = pd.merge(data_ds1000_one, data_ds1000_two, on='error_type', how='outer')
merged_df = pd.merge(merged_df, data_ds1000_three, on='error_type', how='outer')
merged_df = pd.merge(merged_df, data_ds1000_four, on='error_type', how='outer')
merged_df = pd.merge(merged_df, data_ds1000_claude, on='error_type', how='outer')

In [58]:
merged_df.sort_values(by='gpt-4-turbo-2024-04-09', ascending=False)

,error_type,gpt-3.5-turbo-0125,gpt-3.5-turbo-0613,gpt-4-0613,gpt-4-turbo-2024-04-09,claude-3-opus-20240229
23,passed,378.0,378.0,490.0,517.0,38.0
0,AssertionError,366.0,362.0,291.0,278.0,49.0
10,ModuleNotFoundError,50.0,50.0,49.0,50.0,NaN
5,IndentationError,35.0,36.0,41.0,45.0,9.0
21,ValueError,36.0,43.0,37.0,27.0,2.0
19,TypeError,37.0,43.0,23.0,23.0,1.0
11,NameError,29.0,31.0,15.0,13.0,NaN
1,AttributeError,10.0,16.0,8.0,12.0,NaN
8,KeyError,26.0,21.0,25.0,10.0,NaN
17,RuntimeError,5.0,4.0,3.0,6.0,NaN


In [56]:
merged_df.sort_values(by='gpt-4-turbo-2024-04-09', ascending=False).to_csv("results_errors.csv", index=False)

In [85]:
path = 'Data/CoderEval/CodeGen/Raw_Input.jsonl_out.jsonl'
data_codegen = read_jsonl(path, cols=['project','_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['project','_id'])
data_codegen["message"] = data_codegen["eror_msg"].apply(clean_error_msg)
data_codegen["error_type"] = data_codegen["message"].apply(errors_type)
#classify_error_msg(data_codegen)
data_codegen["error_type"] = data_codegen["error_type"].replace('', 'passed')

In [86]:
data_codegen

,project,_id,generate_code,is_pass,eror_msg,message,error_type
0,neo4j/neo4j-python-driver,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 326, in <module>\n raise Exception(""Re...",AssertionError
1,neo4j/neo4j-python-driver,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 324, in <module>\n raise Exception(""Re...",AssertionError
2,neo4j/neo4j-python-driver,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 330, in <module>\n raise Exception(""Re...",AssertionError
3,neo4j/neo4j-python-driver,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 323, in <module>\n raise Exception(""Re...",AssertionError
4,neo4j/neo4j-python-driver,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 326, in <module>\n raise Exception(""Re...",AssertionError
...,...,...,...,...,...,...,...
2290,redhat-openstack/infrared,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,,passed
2291,redhat-openstack/infrared,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,,passed
2292,redhat-openstack/infrared,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,,passed
2293,redhat-openstack/infrared,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",False,/home/travis/builds/repos/redhat-openstack---i...,"line 328, in extend_cli\n extend_cli_root_s...",NameError


In [183]:
path = 'Data/CoderEval/ChatGPT/Raw_Input.jsonl_out.jsonl'
data_gpt= read_jsonl(path, cols=['_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['_id'])
data_gpt["message"] = data_gpt["eror_msg"].apply(clean_error_msg)
classify_error_msg(data_gpt)

,_id,generate_code,is_pass,eror_msg,message,contains_indentation,contains_syntax,contains_unexpected,contains_must,contains_has,contains_name,contains_defined
0,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n r...",False,"Traceback (most recent call last):\n File ""/h...","line 314, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
1,62e60f43d76274f8a4026e28,"from datetime import datetime, timedelta\nfrom...",False,"Traceback (most recent call last):\n File ""/h...","line 347, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
2,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n r...",False,"Traceback (most recent call last):\n File ""/h...","line 314, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
3,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n r...",False,"Traceback (most recent call last):\n File ""/h...","line 314, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
4,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n r...",False,"Traceback (most recent call last):\n File ""/h...","line 314, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2294,62ece4982e6aefcf4aabbd64,"def strip_root(path: str, root: str) -> str:\n...",False,"Traceback (most recent call last):\n File ""/h...","line 12, in test_strip_root\n assert strip_...",False,False,False,False,False,False,False
2295,62ece4982e6aefcf4aabbd64,"def strip_root(path: str, root: str) -> str:\n...",False,"Traceback (most recent call last):\n File ""/h...","line 12, in test_strip_root\n assert strip_...",False,False,False,False,False,False,False
2296,62ece4982e6aefcf4aabbd64,"def strip_root(path: str, root: str) -> str:\n...",False,"Traceback (most recent call last):\n File ""/h...","line 12, in test_strip_root\n assert strip_...",False,False,False,False,False,False,False
2297,62ece4982e6aefcf4aabbd64,"def strip_root(path: str, root: str) -> str:\n...",False,"Traceback (most recent call last):\n File ""/h...","line 12, in test_strip_root\n assert strip_...",False,False,False,False,False,False,False


In [185]:
path = 'Data/CoderEval/PanGu-Coder/Raw_Input.jsonl_out.jsonl'
data_pangu= read_jsonl(path, cols=['_id','generate_code','is_pass','eror_msg'], unlist_cols=['generate_results'], left_cols=['_id'])
data_pangu["message"] = data_pangu["eror_msg"].apply(clean_error_msg)
classify_error_msg(data_pangu)

,_id,generate_code,is_pass,eror_msg,message,contains_indentation,contains_syntax,contains_unexpected,contains_must,contains_has,contains_name,contains_defined
0,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n i...",False,"Traceback (most recent call last):\n File ""/h...","line 324, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
1,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n i...",False,"Traceback (most recent call last):\n File ""/h...","line 317, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
2,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n i...",False,"Traceback (most recent call last):\n File ""/h...","line 317, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
3,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n i...",False,"Traceback (most recent call last):\n File ""/h...","line 317, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
4,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n i...",False,"Traceback (most recent call last):\n File ""/h...","line 322, in <module>\n raise Exception(""Re...",False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
2295,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n pa...",True,/home/travis/builds/repos/redhat-openstack---i...,,False,False,False,False,False,False,False
2296,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n ex...",False,/home/travis/builds/repos/redhat-openstack---i...,"line 323, in extend_cli\n extend_cli_base(s...",False,False,False,False,False,True,True
2297,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n cl...",False,/home/travis/builds/repos/redhat-openstack---i...,"line 323, in extend_cli\n cli_opts = self._...",False,False,False,False,True,False,False
2298,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n ro...",False,/home/travis/builds/repos/redhat-openstack---i...,"line 326, in extend_cli\n root_subparser.se...",False,False,False,False,True,False,False


In [186]:
data_pangu[['contains_indentation', 'contains_syntax', 'contains_must', 'contains_has', 'contains_unexpected', 'contains_name']].sum()

contains_indentation      0
contains_syntax           1
contains_must            23
contains_has            342
contains_unexpected      21
contains_name           619
dtype: int64

In [193]:
for i, m in enumerate(data_pangu[data_pangu["contains_must"] == 1]["message"]):
    print(f"{i}:\n{m}")

0:
line 85, in <listcomp>
    if not any([o.startswith(seen) for o in ordering]):
TypeError: startswith first arg must be str or a tuple of str, not set

1:
line 177, in _get_err_indices
    coord = self.coords[coord_name]
TypeError: list indices must be integers or slices, not str

2:
line 60, in difference
    return dict(d1, **d2 if level < 0 else d1.copy())
TypeError: dict() argument after ** must be a mapping, not NoneType

3:
line 318, in update_last_applied_manifest_list_from_resp
    last_applied_manifest[k] = response[k]
TypeError: list indices must be integers or slices, not dict

4:
line 318, in update_last_applied_manifest_list_from_resp
    last_applied_manifest[field] = response[field]
TypeError: list indices must be integers or slices, not dict

5:
line 318, in update_last_applied_manifest_list_from_resp
    last_applied_manifest[field] = observer_schema[field]
TypeError: list indices must be integers or slices, not dict

6:
line 318, in update_last_applied_manifest_list

In [165]:
data_codegen

,_id,generate_code,is_pass,eror_msg,message
0,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 326, in <module>\n raise Exception(""Re..."
1,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 324, in <module>\n raise Exception(""Re..."
2,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 330, in <module>\n raise Exception(""Re..."
3,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 323, in <module>\n raise Exception(""Re..."
4,62e60f43d76274f8a4026e28,"def hydrate_time(nanoseconds, tz=None):\n\t""""""...",False,"Traceback (most recent call last):\n File ""/h...","line 326, in <module>\n raise Exception(""Re..."
...,...,...,...,...,...
2290,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,
2291,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,
2292,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",True,/home/travis/builds/repos/redhat-openstack---i...,
2293,63060b1b73426c38ae68ad43,"def extend_cli(self, root_subparsers):\n\t""""""\...",False,/home/travis/builds/repos/redhat-openstack---i...,"line 328, in extend_cli\n extend_cli_root_s..."


In [58]:
path = 'Data/DS-1000/ds1000_data/Scipy/Completion/q2/input/input1.pkl'

In [59]:
with open(path, 'rb') as f:
    data = pickle.load(f)
type(data)

tuple

In [60]:
print(data)

(array([10, 19, 30, 35, 51], dtype=int32), array([ 1,  7, 20, 50, 79], dtype=int32), (4, 0.1, 1))


In [81]:
path = 'Data/CoderEval/experimental-results/GenerateResultPython/ChatGPT/Raw_Input.jsonl'

In [82]:
data = []
with open(path, 'r') as file:
    for line in file:
        data.append(json.loads(line))

In [83]:
print(len(data))

230


In [84]:
data[0]

{'_id': '62e60f43d76274f8a4026e28',
 'generate_results': ['def hydrate_time(nanoseconds, tz=None):\n    return Time(nanoseconds, tz)',
  'from datetime import datetime, timedelta\nfrom typing import Optional\n\nclass Time:\n    def __init__(self, hour: int, minute: int, second: int, microsecond: int):\n        self.hour = hour\n        self.minute = minute\n        self.second = second\n        self.microsecond = microsecond\n\nclass LocalTime(Time):\n    pass\n\ndef hydrate_time(nanoseconds: int, tz: Optional[str] = None) -> Time:\n    """\n    Hydrator for `Time` and `LocalTime` values.\n\n    :param nanoseconds: Number of nanoseconds.\n    :param tz: Timezone string (optional).\n    :return: Time object.\n    """\n    microseconds = nanoseconds // 1000\n    seconds, microseconds = divmod(microseconds, 1000000)\n    minutes, seconds = divmod(seconds, 60)\n    hours, minutes = divmod(minutes, 60)\n    if tz:\n        dt = datetime.now()\n        delta = timedelta(hours=hours, minutes=

In [90]:
print(data[0]["generate_results"][1])

from datetime import datetime, timedelta
from typing import Optional

class Time:
    def __init__(self, hour: int, minute: int, second: int, microsecond: int):
        self.hour = hour
        self.minute = minute
        self.second = second
        self.microsecond = microsecond

class LocalTime(Time):
    pass

def hydrate_time(nanoseconds: int, tz: Optional[str] = None) -> Time:
    """
    Hydrator for `Time` and `LocalTime` values.

    :param nanoseconds: Number of nanoseconds.
    :param tz: Timezone string (optional).
    :return: Time object.
    """
    microseconds = nanoseconds // 1000
    seconds, microseconds = divmod(microseconds, 1000000)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if tz:
        dt = datetime.now()
        delta = timedelta(hours=hours, minutes=minutes, seconds=seconds, microseconds=microseconds)
        if tz == 'UTC':
            dt = dt.replace(tzinfo=None)
        else:
            dt = dt.astimezone(py

In [80]:
data[0]

{'file_path': 'neo4j/_codec/hydration/v1/temporal.py',
 'project': 'neo4j/neo4j-python-driver',
 'name': 'hydrate_time',
 'docstring': 'Hydrator for `Time` and `LocalTime` values.\n\n:param nanoseconds:\n:param tz:\n:return: Time',
 '_id': '62e60f43d76274f8a4026e28',
 'code': ['def hydrate_time(nanoseconds, tz=None):\n    return Time(nanoseconds, tz)',
  'from datetime import datetime, timedelta\nfrom typing import Optional\n\nclass Time:\n    def __init__(self, hour: int, minute: int, second: int, microsecond: int):\n        self.hour = hour\n        self.minute = minute\n        self.second = second\n        self.microsecond = microsecond\n\nclass LocalTime(Time):\n    pass\n\ndef hydrate_time(nanoseconds: int, tz: Optional[str] = None) -> Time:\n    """\n    Hydrator for `Time` and `LocalTime` values.\n\n    :param nanoseconds: Number of nanoseconds.\n    :param tz: Timezone string (optional).\n    :return: Time object.\n    """\n    microseconds = nanoseconds // 1000\n    seconds, 

In [78]:
data[0]["generate_results"]

[{'generate_code': 'def hydrate_time(nanoseconds, tz=None):\n    return Time(nanoseconds, tz)',
  'is_pass': False,
  'return_code': 1},
 {'generate_code': 'from datetime import datetime, timedelta\nfrom typing import Optional\n\nclass Time:\n    def __init__(self, hour: int, minute: int, second: int, microsecond: int):\n        self.hour = hour\n        self.minute = minute\n        self.second = second\n        self.microsecond = microsecond\n\nclass LocalTime(Time):\n    pass\n\ndef hydrate_time(nanoseconds: int, tz: Optional[str] = None) -> Time:\n    """\n    Hydrator for `Time` and `LocalTime` values.\n\n    :param nanoseconds: Number of nanoseconds.\n    :param tz: Timezone string (optional).\n    :return: Time object.\n    """\n    microseconds = nanoseconds // 1000\n    seconds, microseconds = divmod(microseconds, 1000000)\n    minutes, seconds = divmod(seconds, 60)\n    hours, minutes = divmod(minutes, 60)\n    if tz:\n        dt = datetime.now()\n        delta = timedelta(ho

In [102]:
str_code = "def parse_arguments(*arguments):\n\t\"\"\"\n\tGiven command-line arguments with which this script was invoked, parse the arguments and return\nthem as an ArgumentParser instance.\n\t\"\"\"\n\tfrom argparse import ArgumentParser, ArgumentTypeError\n\n\tparser = ArgumentParser(description='A tool to generate a Python file from an XML document.')\n\tparser.add_argument('-d', '--doc', dest='doc', help='The XML document to be parsed.', type=str, required=True)\n\tparser.add_argument('-o', '--output', dest='output', help='The path to the generated Python file.', type=str, required=True)\n\tparser.add_argument('-f', '--force', dest='force', help='Force overwrite an existing Python file.', action='store_true', required=False)\n\tparser.add_argument('-l', '--list', dest='list', help='List all the files in the specified directory.', action='store_true', required=False)\n\tparser.add_argument('-v', '--verbose', dest='verbose', help='Verbose mode.', action='store_true', required=False)\n\tparser.add_argument('-q', '--quiet', dest='quiet', help='Quiet mode.', action='store_true', required=False)\n\tparser.add_argument('-n', '--no-exclude', dest='no_exclude', help='Do not exclude an XML file.', action='store_true', required=False)\n\tparser.add_argument('-t', '--type', dest='type', help='The type of XML file to be generated.', type=str, required=True, choices=['xml', 'python'])\n\tgroup = parser.add_mutually_exclusive_group(required=False)\n\tgroup.add_argument('-e', '--exclude', dest='exclude', help='Exclude an XML file.', action='store_true', required=False)\n\tgroup.add_argument('-p', '--parse', dest='parse', help='Parse an XML file.', action='store_true', required=False)\n\targs = parser.parse_args(*arguments)\n\n\tif args.type == 'xml' and not args.doc:\n\t\tparser.error('A XML file must be specified.')\n\n\tif args.verbose:\n\t\targs.quiet = True\n\t\targs.verbose = True\n\tif args.quiet:\n\t\targs.verbose = False\n\t\targs.quiet = False\n\n\treturn args#\n\nparse_arguments()"

In [103]:
print(str_code)

def parse_arguments(*arguments):
	"""
	Given command-line arguments with which this script was invoked, parse the arguments and return
them as an ArgumentParser instance.
	"""
	from argparse import ArgumentParser, ArgumentTypeError

	parser = ArgumentParser(description='A tool to generate a Python file from an XML document.')
	parser.add_argument('-d', '--doc', dest='doc', help='The XML document to be parsed.', type=str, required=True)
	parser.add_argument('-o', '--output', dest='output', help='The path to the generated Python file.', type=str, required=True)
	parser.add_argument('-f', '--force', dest='force', help='Force overwrite an existing Python file.', action='store_true', required=False)
	parser.add_argument('-l', '--list', dest='list', help='List all the files in the specified directory.', action='store_true', required=False)
	parser.add_argument('-v', '--verbose', dest='verbose', help='Verbose mode.', action='store_true', required=False)
	parser.add_argument('-q', '--quiet', d

In [104]:
exec(str_code)

usage: ipykernel_launcher.py [-h] -d DOC -o OUTPUT [-f] [-l] [-v] [-q] [-n] -t
                             {xml,python} [-e | -p]
ipykernel_launcher.py: error: the following arguments are required: -d/--doc, -o/--output, -t/--type


SystemExit: 2

/opt/homebrew/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3534: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [106]:
def sumx(x=1, y=2):
    return x+y#
    

In [107]:
sumx()

3